In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/pinxau1000/radioml2018/datasets.desktop
/kaggle/input/datasets/pinxau1000/radioml2018/classes-fixed.json
/kaggle/input/datasets/pinxau1000/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5
/kaggle/input/datasets/pinxau1000/radioml2018/classes-fixed.txt
/kaggle/input/datasets/pinxau1000/radioml2018/LICENSE.TXT
/kaggle/input/datasets/pinxau1000/radioml2018/classes.txt
/kaggle/input/datasets/venugopalbattula/5-1-checkpoints/latest_ckpt_phase1_ncritic5 (4).pt
/kaggle/input/datasets/venugopalbattula/5-1-checkpoints/latest_ckpt_phase1_ncritic5 (13).pt
/kaggle/input/datasets/venugopalbattula/5-1-checkpoints/latest_ckpt_phase1_ncritic5 (10).pt
/kaggle/input/datasets/venugopalbattula/5-1-checkpoints/latest_ckpt_phase1_ncritic5 (7).pt


# 05.1 — GAN Training (Controlled Replication)

This notebook is a controlled re-run of Phase 1 training, correcting three
compute-driven deviations from the paper that were present in `05_train_gan.ipynb`:

| Deviation | Old (05) | New (05.1) |
|-----------|----------|------------|
| n_critic | 2 | 5 (paper spec) |
| Samples per (class, SNR) cell | 1024 | 2048 |
| Phase 1 epochs | 40 | 100 |

Goal: determine whether the mode collapse observed in the original run is
attributable to these compute compromises, or is a genuine limitation of the
static cWGAN-GP method. A live diversity monitor (intra-class cosine
similarity + per-sample std ratio) is logged every epoch so collapse is
visible during training, not just at final evaluation.

Checkpoints are saved to `/kaggle/working/checkpoints/` every epoch AND
mirrored to `/kaggle/working/latest_ckpt_phase1_ncritic5.pt` every epoch,
so the Output tab always shows the most recent state even if the session
crashes mid-run.

## 1. Setup — pull source modules from GitHub

In [2]:
import os, sys

REPO    = "glitching-gops/lpd-radar-waveform"
BRANCH  = "main"
BASE    = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/src"
SRC_DIR = "/kaggle/working/src"
os.makedirs(SRC_DIR, exist_ok=True)

for fname in ["__init__.py", "dataset.py", "models.py", "ambiguity.py"]:
    ret = os.system(f"wget -q -O {SRC_DIR}/{fname} {BASE}/{fname}")
    print(f"  {fname:<15} {'✓' if ret == 0 else '✗ FAILED'}")

sys.path.insert(0, "/kaggle/working")

  __init__.py     ✓
  dataset.py      ✓
  models.py       ✗ FAILED
  ambiguity.py    ✓


In [3]:
import time
import shutil
import numpy as np
import torch
import torch.nn as nn
import wandb
from kaggle_secrets import UserSecretsClient

from src.dataset   import get_loaders, MODULATIONS, SNR_VALUES
from src.models    import Generator, Critic
from src.ambiguity import AmbiguityLoss

print("Imports ready ✓")

ImportError: cannot import name 'Generator' from 'src.models' (/kaggle/working/src/models.py)

In [ ]:
# ── device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── dataset path ──────────────────────────────────────────────────────────────
HDF5_PATH = None
for dirpath, _, filenames in os.walk("/kaggle/input"):
    for fname in filenames:
        if fname.endswith(".hdf5") or fname.endswith(".h5"):
            HDF5_PATH = os.path.join(dirpath, fname)

assert HDF5_PATH is not None, "HDF5 file not found under /kaggle/input"
print(f"Dataset : {HDF5_PATH}")

# ── checkpoint dirs ───────────────────────────────────────────────────────────
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Checkpoints : {CKPT_DIR}")

# ── previous run checkpoint (for comparison, not resuming from) ─────────────
PHASE1_OLD_CKPT = "/kaggle/input/lpd-radar-checkpoints/phase1_final_epoch39.pt"
print(f"Old run checkpoint (reference only) : {PHASE1_OLD_CKPT}")

## 2. Hyperparameters — controlled replication config

In [ ]:
# ── Phase 1: controlled replication ───────────────────────────────────────────
P1 = dict(
    phase          = 1,
    n_epochs       = 100,       # full run — paper-scale training
    batch_size     = 512,
    lr_G           = 1e-5,      # Adam  (paper)
    lr_C           = 5e-5,      # RMSProp (paper)
    n_critic       = 5,         # ← restored to paper spec (was 2)
    lambda_gp      = 10,
    latent_dim     = 512,
    eta            = 0.0,       # no ambiguity loss in Phase 1
)

# ── Phase 2 config kept here for later use, unchanged ─────────────────────────
P2 = dict(
    phase          = 2,
    n_epochs       = 10,
    batch_size     = 512,
    lr_G           = 1e-5,
    lr_C           = 5e-5,
    n_critic       = 5,
    lambda_gp      = 10,
    latent_dim     = 512,
    eta            = 1e-4,
    n_doppler      = 256,
    target_peak    = 1000.0,
    zero_doppler_w = 1000.0,
    n_delays       = 128,
)

SAMPLES_PER_CELL = 2048   # ← doubled from 1024

print("Phase 1 config:", P1)
print(f"Samples per (class, SNR) cell: {SAMPLES_PER_CELL}")

## 3. Weights & Biases setup

In [ ]:
def init_wandb(config: dict, run_name: str) -> None:
    """Initialise a W&B run with the given config."""
    secret  = UserSecretsClient()
    api_key = secret.get_secret("WANDB_API_KEY")
    wandb.login(key=api_key, relogin=False)

    wandb.init(
        project = "lpd-radar-thesis",
        name    = run_name,
        config  = config,
        reinit  = True,
    )
    print(f"W&B run: {wandb.run.name}  |  {wandb.run.url}")

## 4. Data loaders — 2048 samples per (class, SNR) cell

Doubling the subset from 1024 to 2048 samples per cell increases RAM usage
from ~2.6 GB to ~5.2 GB. Still well within Kaggle's ~29 GB limit.

In [ ]:
print("Building data loaders (2048 samples/cell)...")
loader_gen, loader_det, meta = get_loaders(
    HDF5_PATH,
    batch_size=P1["batch_size"],
    samples_per_cell=SAMPLES_PER_CELL,
    cache=True,
)
print(f"Generator loader : {len(loader_gen):,} batches/epoch")
print(f"Detector  loader : {len(loader_det):,} batches/epoch")

## 5. cWGAN-GP loss components

In [ ]:
def gradient_penalty(critic: nn.Module,
                     x_real: torch.Tensor,
                     x_fake: torch.Tensor,
                     y: torch.Tensor,
                     lambda_gp: float) -> torch.Tensor:
    """WGAN-GP gradient penalty on interpolated samples."""
    B = x_real.size(0)
    alpha = torch.rand(B, 1, 1, device=x_real.device)
    x_hat = (alpha * x_real + (1 - alpha) * x_fake).requires_grad_(True)

    score_hat = critic(x_hat, y)
    gradients = torch.autograd.grad(
        outputs      = score_hat,
        inputs       = x_hat,
        grad_outputs = torch.ones_like(score_hat),
        create_graph = True,
        retain_graph = True,
    )[0]

    grad_norm = gradients.view(B, -1).norm(2, dim=1)
    return lambda_gp * ((grad_norm - 1) ** 2).mean()

In [ ]:
def critic_step(G, C, opt_C, x_real, y, cfg):
    """One critic update. Returns dict of loss components."""
    B = x_real.size(0)
    z = torch.randn(B, cfg["latent_dim"], 1, device=x_real.device)

    with torch.no_grad():
        x_fake = G(z, y)

    score_real = C(x_real, y)
    score_fake = C(x_fake, y)
    w_dist = score_real.mean() - score_fake.mean()

    gp = gradient_penalty(C, x_real, x_fake.detach(), y, cfg["lambda_gp"])
    loss_C = -w_dist + gp

    opt_C.zero_grad()
    loss_C.backward()
    opt_C.step()

    return {
        "loss_C":     loss_C.item(),
        "w_dist":     w_dist.item(),
        "gp":         gp.item(),
        "score_real": score_real.mean().item(),
        "score_fake": score_fake.mean().item(),
    }


def generator_step(G, C, opt_G, y, cfg, ambiguity_fn=None):
    """One generator update. Returns dict of loss components."""
    B = y.size(0)
    z = torch.randn(B, cfg["latent_dim"], 1, device=y.device)

    x_fake     = G(z, y)
    score_fake = C(x_fake, y)
    loss_G     = -score_fake.mean()

    loss_ambig = torch.tensor(0.0, device=y.device)
    if ambiguity_fn is not None and cfg["eta"] > 0:
        loss_ambig = ambiguity_fn(x_fake)
        loss_G     = loss_G + cfg["eta"] * loss_ambig

    opt_G.zero_grad()
    loss_G.backward()
    opt_G.step()

    return {"loss_G": loss_G.item(), "loss_ambig": loss_ambig.item()}

## 6. Diversity monitor

Measures mode collapse live, on a **fixed** monitoring batch created once
before training starts. Two metrics, both computed on unit-std normalised
waveforms (same normalisation used in evaluation):

- **std_ratio** — background per-sample std ÷ generated per-sample std.
  Close to 1.0 is healthy. The collapsed run measured ~1.4.
- **cos_ratio** — generated intra-class cosine similarity ÷ background
  intra-class cosine similarity. Close to 1.0 is healthy. The collapsed
  run measured ~14×.

In [ ]:
@torch.no_grad()
def measure_diversity(G, monitor_z, monitor_y, background_ref):
    """
    Returns dict of diversity metrics for the current generator,
    evaluated on a fixed monitoring batch.
    """
    G.eval()
    x_gen = G(monitor_z, monitor_y)

    def unit_std(x):
        m = x.mean(dim=2, keepdim=True)
        s = x.std(dim=2, keepdim=True) + 1e-8
        return (x - m) / s

    g = unit_std(x_gen)
    b = unit_std(background_ref.to(x_gen.device))

    gen_std = g.std(dim=0).mean().item()
    bg_std  = b.std(dim=0).mean().item()

    gf  = g.flatten(1)
    gf  = gf / (gf.norm(dim=1, keepdim=True) + 1e-8)
    cos = gf @ gf.T
    n   = cos.size(0)
    gen_cos = (cos.sum() - cos.diag().sum()).item() / (n * (n - 1))

    bf   = b.flatten(1)
    bf   = bf / (bf.norm(dim=1, keepdim=True) + 1e-8)
    cosb = bf @ bf.T
    nb   = cosb.size(0)
    bg_cos = (cosb.sum() - cosb.diag().sum()).item() / (nb * (nb - 1))

    G.train()
    return {
        "gen_std":   gen_std,
        "bg_std":    bg_std,
        "std_ratio": bg_std / (gen_std + 1e-8),
        "gen_cos":   gen_cos,
        "bg_cos":    bg_cos,
        # difference instead of ratio — stable even when bg_cos ≈ 0
        "cos_gap":   gen_cos - bg_cos,
    }

In [ ]:
# ── fixed monitoring inputs — created ONCE, reused every epoch ───────────────
torch.manual_seed(1234)
MONITOR_N = 256

monitor_z = torch.randn(MONITOR_N, P1["latent_dim"], 1, device=DEVICE)
_mon_y, _, _ = next(iter(loader_gen))
monitor_y = _mon_y[:MONITOR_N].to(DEVICE)

_bg_ref, _, _ = next(iter(loader_gen))
background_ref = _bg_ref[:MONITOR_N]

print(f"Monitoring batch fixed at {MONITOR_N} samples ✓")

## 7. Checkpoint utilities

Every epoch, the checkpoint is saved twice:
1. To `/kaggle/working/checkpoints/` with the epoch number in the filename
   (full history, for resuming from any point)
2. To `/kaggle/working/latest_ckpt_phase1_ncritic5.pt` (overwritten every
   epoch — this is the file to download if the session crashes)

In [ ]:
def save_checkpoint(G, C, opt_G, opt_C, epoch, metrics, tag, ckpt_dir):
    """Save checkpoint to numbered file AND mirror to a fixed 'latest' file."""
    path = os.path.join(ckpt_dir, f"ckpt_{tag}_epoch{epoch:04d}.pt")
    metrics_clean = {k: float(v) for k, v in metrics.items()}
    torch.save({
        "epoch":   int(epoch),
        "metrics": metrics_clean,
        "G":       G.state_dict(),
        "C":       C.state_dict(),
        "opt_G":   opt_G.state_dict(),
        "opt_C":   opt_C.state_dict(),
    }, path)

    # mirror to a fixed filename in /kaggle/working/ root — always visible
    # in the Output tab, always the most recent epoch, overwritten every time
    latest_path = f"/kaggle/working/latest_ckpt_{tag}.pt"
    shutil.copy(path, latest_path)

    return path


def load_checkpoint(path, G, C, opt_G=None, opt_C=None):
    """Load checkpoint. weights_only=False required — metrics contain numpy scalars."""
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    G.load_state_dict(ckpt["G"])
    C.load_state_dict(ckpt["C"])
    if opt_G is not None:
        opt_G.load_state_dict(ckpt["opt_G"])
    if opt_C is not None:
        opt_C.load_state_dict(ckpt["opt_C"])
    print(f"Loaded checkpoint: {path}  (epoch {ckpt['epoch']})")
    return ckpt["epoch"], ckpt["metrics"]

## 8. Training loop

Same structure as the original, with the diversity monitor hooked in at
the end of every epoch and logged to W&B alongside the standard losses.

In [ ]:
def train(cfg: dict,
          G: nn.Module,
          C: nn.Module,
          loader_gen,
          run_name: str,
          monitor_z: torch.Tensor,
          monitor_y: torch.Tensor,
          background_ref: torch.Tensor,
          ambiguity_fn = None,
          resume_path: str = None,
          tag: str = "phase1_ncritic5"):
    """
    Full training loop with live diversity monitoring.
    """
    init_wandb(cfg, run_name)

    opt_G = torch.optim.Adam(G.parameters(), lr=cfg["lr_G"], betas=(0.0, 0.9))
    opt_C = torch.optim.RMSprop(C.parameters(), lr=cfg["lr_C"])

    start_epoch = 0
    if resume_path:
        start_epoch, _ = load_checkpoint(resume_path, G, C, opt_G, opt_C)
        start_epoch += 1

    best_w_dist = -float("inf")
    best_ckpt   = None

    for epoch in range(start_epoch, cfg["n_epochs"]):
        G.train()
        C.train()

        epoch_metrics = {
            "loss_C": [], "w_dist": [], "gp": [],
            "score_real": [], "score_fake": [],
            "loss_G": [], "loss_ambig": [],
        }

        data_iter = iter(loader_gen)
        step      = 0
        t_epoch   = time.time()

        while True:
            critic_logs = []
            for _ in range(cfg["n_critic"]):
                try:
                    x_real, _, _ = next(data_iter)
                except StopIteration:
                    data_iter = iter(loader_gen)
                    x_real, _, _ = next(data_iter)
                x_real = x_real.to(DEVICE)

                try:
                    y_batch, _, _ = next(data_iter)
                except StopIteration:
                    data_iter = iter(loader_gen)
                    y_batch, _, _ = next(data_iter)
                y_batch = y_batch.to(DEVICE)

                log = critic_step(G, C, opt_C, x_real, y_batch, cfg)
                critic_logs.append(log)

            for k in ["loss_C", "w_dist", "gp", "score_real", "score_fake"]:
                epoch_metrics[k].append(np.mean([l[k] for l in critic_logs]))

            try:
                y_batch, _, _ = next(data_iter)
                gen_log = generator_step(G, C, opt_G, y_batch.to(DEVICE), cfg, ambiguity_fn)
                for k in ["loss_G", "loss_ambig"]:
                    epoch_metrics[k].append(gen_log[k])
                step += 1
            except StopIteration:
                data_iter = iter(loader_gen)
                y_batch, _, _ = next(data_iter)
                gen_log = generator_step(G, C, opt_G, y_batch.to(DEVICE), cfg, ambiguity_fn)
                for k in ["loss_G", "loss_ambig"]:
                    epoch_metrics[k].append(gen_log[k])
                step += 1
                break

            if step % 50 == 0:
                wandb.log({
                    "step/loss_C": critic_logs[-1]["loss_C"],
                    "step/loss_G": gen_log["loss_G"],
                    "step/w_dist": critic_logs[-1]["w_dist"],
                    "step/gp":     critic_logs[-1]["gp"],
                    "epoch":       epoch,
                })

        elapsed = time.time() - t_epoch
        mean    = {k: np.mean(v) for k, v in epoch_metrics.items()}

        # ── diversity monitor — MUST run before the print/log below ─────────
        div = measure_diversity(G, monitor_z, monitor_y, background_ref)

        print(
            f"Epoch {epoch:04d} | W-dist: {mean['w_dist']:+.4f} | "
            f"loss_C: {mean['loss_C']:+.4f} | loss_G: {mean['loss_G']:+.4f} | "
            f"GP: {mean['gp']:.4f} | time: {elapsed:.0f}s"
        )
        print(
            f"           diversity | std_ratio: {div['std_ratio']:.3f}  "
            f"cos_gap: {div['cos_gap']:+.4f}  "
            f"(gen_cos {div['gen_cos']:+.4f} vs bg_cos {div['bg_cos']:+.4f})  "
            f"(gen_std {div['gen_std']:.3f} vs bg {div['bg_std']:.3f})"
        )

        wandb.log({
            "epoch/w_dist":     mean["w_dist"],
            "epoch/loss_C":     mean["loss_C"],
            "epoch/loss_G":     mean["loss_G"],
            "epoch/gp":         mean["gp"],
            "epoch/loss_ambig": mean["loss_ambig"],
            "epoch/std_ratio":  div["std_ratio"],
            "epoch/cos_gap":    div["cos_gap"],
            "epoch/gen_cos":    div["gen_cos"],
            "epoch/bg_cos":     div["bg_cos"],
            "epoch/gen_std":    div["gen_std"],
            "epoch":            epoch,
        })

        ckpt_path = save_checkpoint(G, C, opt_G, opt_C, epoch, mean, tag=tag, ckpt_dir=CKPT_DIR)

        if mean["w_dist"] > best_w_dist:
            best_w_dist = mean["w_dist"]
            best_ckpt   = ckpt_path

    print(f"\nTraining complete. Best W-dist: {best_w_dist:.4f}")
    print(f"Best checkpoint: {best_ckpt}")
    print(f"Latest checkpoint (always current): /kaggle/working/latest_ckpt_{tag}.pt")
    wandb.finish()
    return best_ckpt

## 10. Run Phase 1 — 100 epochs, n_critic=5, 2048 samples/cell

At n_critic=5 the critic runs ~2.5× more steps per epoch than the original
run. Expect roughly 20–25 minutes per epoch. Budget ~4–5 sessions of
~20 epochs each to reach 100.

**To resume a later session:**
```python
resume_path = "/kaggle/input/lpd-radar-checkpoints/latest_ckpt_phase1_ncritic5.pt"
```
(after uploading the latest mirrored checkpoint to your Kaggle dataset)

In [ ]:
G = Generator(latent_dim=P1["latent_dim"]).to(DEVICE)
C = Critic().to(DEVICE)

# Set resume_path to continue from a previous session's checkpoint.
# Leave as None to start this controlled run fresh.
RESUME_PATH = "/kaggle/input/datasets/venugopalbattula/5-1-checkpoints/latest_ckpt_phase1_ncritic5 (13).pt"

PHASE1_NCRITIC5_BEST = train(
    cfg            = P1,
    G              = G,
    C              = C,
    loader_gen     = loader_gen,
    run_name       = "phase1-ncritic5-controlled",
    monitor_z      = monitor_z,
    monitor_y      = monitor_y,
    background_ref = background_ref,
    ambiguity_fn   = None,
    resume_path    = RESUME_PATH,
    tag            = "phase1_ncritic5",
)

print(f"\nBest checkpoint: {PHASE1_NCRITIC5_BEST}")

In [ ]:
import os, torch
path = "/kaggle/working/latest_ckpt_phase1_ncritic5.pt"
print(os.path.exists(path))
ckpt = torch.load(path, map_location="cpu", weights_only=False)
print("epoch:", ckpt["epoch"], "| w_dist:", ckpt["metrics"]["w_dist"])